# CLINC150 -> bộ dữ liệu intent tiếng Việt (20 intent ngân hàng/thẻ)

**Quy trình:** lọc 20 intent -> tạo danh sách câu -> dịch **thủ công qua web chat** (ChatGPT / Gemini web / Claude web, không cần API) -> gộp, kiểm tra, khử trùng lặp -> lưu `subset20_vi.json`.

**Chuẩn bị:**
1. Upload `data_full.json` (từ repo clinc/oos-eval, thư mục `data/`) vào panel Tệp của Colab.
2. Không cần API key.
3. Chạy lần lượt từng cell. Mặc định `TEST_MODE = True` (chỉ dịch 120 câu để kiểm tra chất lượng). Hài lòng thì đổi thành `False`.

**Lưu ý:** dùng cùng một web chat cho cả bộ dữ liệu để văn phong nhất quán. Tải thư mục `batches_vi/` về định kỳ vì Colab xóa file khi hết phiên.

In [13]:
# Không cần cài thêm thư viện nào (không dùng API)
print('OK')

OK


## 1. Cấu hình

In [14]:
import json, os, random, re, time, glob
from collections import Counter

DATA_FILE   = "data_full.json"
CHUNK       = 100                  # số câu mỗi lần dán vào web chat (lô dài dễ bị bỏ sót dòng)
TEST_MODE   = False                # True: chỉ dịch 120 câu mẫu. Kiểm tra chất lượng xong thì đổi False
N_OOS_TEST  = 300                  # số câu OOS test lấy mẫu để dịch (gốc có 1000)
SEED        = 42

KEEP = {
    # ngân hàng (13)
    "account_blocked", "balance", "bill_balance", "bill_due", "freeze_account",
    "interest_rate", "min_payment", "pay_bill", "pin_change", "report_fraud",
    "spending_history", "transactions", "transfer",
    # thẻ (7)
    "card_declined", "credit_limit", "credit_score", "expiration_date",
    "international_fees", "new_card", "report_lost_card",
}
assert len(KEEP) == 20, len(KEEP)

## 2. Đọc dữ liệu gốc và kiểm tra tên intent

In [15]:
d = json.load(open(DATA_FILE))
print(d.keys())
cnt = Counter(x[1] for x in d["train"])
print(len(cnt), "intent")

missing = KEEP - set(cnt)
assert not missing, f"Sai tên intent: {missing}"
print("OK: 20 intent đều có trong dữ liệu")

dict_keys(['oos_val', 'val', 'train', 'oos_test', 'test', 'oos_train'])
150 intent
OK: 20 intent đều có trong dữ liệu


## 3. Lọc 20 intent, lấy mẫu OOS test, tạo danh sách câu cần dịch

In [16]:
random.seed(SEED)

out = {k: [x for x in d[k] if x[1] in KEEP] for k in ["train", "val", "test"]}
for k in ["oos_train", "oos_val", "oos_test"]:
    out[k] = d[k]
json.dump(out, open("subset20_en_full.json", "w"), indent=1)

out["oos_test"] = random.sample(out["oos_test"], N_OOS_TEST)
json.dump(out, open("subset20_en.json", "w"), indent=1)
print({k: len(v) for k, v in out.items()})

# mỗi câu có id duy nhất: <tập>_<số thứ tự>
items = []
for split in ["train", "val", "test", "oos_train", "oos_val", "oos_test"]:
    for i, (text, label) in enumerate(out[split]):
        items.append({"id": f"{split}_{i}", "split": split, "en": text, "label": label})
json.dump(items, open("items_en.json", "w"), indent=1)
print("tổng", len(items), "câu")

{'train': 2000, 'val': 400, 'test': 600, 'oos_train': 100, 'oos_val': 100, 'oos_test': 300}
tổng 3500 câu


## 4. Dịch thủ công qua web chat

Mỗi vòng gồm 2 bước:
1. Chạy `next_chunk()` -> copy toàn bộ phần prompt + câu được in ra -> dán vào web chat -> copy kết quả.
2. Dán kết quả vào biến `RESULT` rồi chạy `save_result(RESULT)`.

Lặp lại đến khi `next_chunk()` báo "Xong!". Câu bị bỏ sót sẽ tự được đưa vào lô sau, không cần làm gì thêm.

In [93]:
PROMPT = """Dịch các câu tiếng Anh sau sang tiếng Việt, theo cách người Việt thật sự nhắn tin cho trợ lý ngân hàng/thẻ.
Quy tắc:
- Chữ thường, giữ dấu tiếng Việt, không dấu câu cuối câu. Dịch theo ý, không dịch từng chữ.
- Tiền: quy đổi sang VND với số hợp lý (nhân khoảng 20-25 nghìn rồi làm tròn cho tự nhiên) và viết như người Việt: 10 dollars -> 200k, 100 dollars -> 2 triệu, 1200 -> 25 triệu, 20000 -> 500 triệu. Nếu là ký hiệu chưa xác định như "$x" thì giữ "x".
- Tên ngân hàng/ví/thẻ Mỹ: thay bằng tên Việt Nam, nhất quán cho cùng một tên: bank of america -> vietcombank, boa -> vietcombank, capital one -> techcombank, usaa -> mb bank, navy federal -> acb, ally -> vpbank, chase -> bidv, wells fargo -> agribank, paypal -> momo, bluebird -> zalopay, discover -> sacombank, amex -> vib, citibank -> hdbank, washington federal -> ocb, barclay's -> tpbank, apollo -> shb, western -> eximbank.
- Thuật ngữ cố định: checking account -> tài khoản thanh toán; savings account -> tài khoản tiết kiệm; credit card -> thẻ tín dụng; debit card -> thẻ ghi nợ; balance -> số dư; APR / annual percentage rate -> lãi suất hàng năm; report lost card -> báo mất thẻ; stolen -> bị mất cắp; utility bills -> hóa đơn điện nước.
- Xưng hô: đa số câu bỏ chủ ngữ hoặc dùng "mình"/"tôi"; chỉ khoảng 1 trong 5 câu dùng "em"; gọi trợ lý là "bạn" hoặc bỏ. Dùng "chuyển/gửi", không dùng "cho" thay cho chuyển.
- Đầu ra: mỗi dòng đúng dạng `id | câu tiếng Việt`, giữ nguyên id và thứ tự, không giải thích, không bỏ dòng nào. Chỉ trả về các dòng, không bọc trong code block.

"""

LINE_RE = re.compile(r"^\s*([a-z_]+_\d+)\s*\|\s*(.+?)\s*$")
os.makedirs("batches_vi", exist_ok=True)

if TEST_MODE:
    rng = random.Random(SEED)
    scope = []
    for label in sorted(KEEP):     # 5 câu train mỗi intent = 100 câu
        pool = [it for it in items if it["split"] == "train" and it["label"] == label]
        scope += rng.sample(pool, 5)
    scope += rng.sample([it for it in items if it["split"] == "oos_train"], 20)   # + 20 câu OOS
else:
    scope = items[:]
    random.Random(SEED).shuffle(scope)
print("sẽ dịch", len(scope), "câu")

def load_translations():
    vi = {}
    for fn in sorted(glob.glob("batches_vi/*.txt")):
        for line in open(fn, encoding="utf-8"):
            m = LINE_RE.match(line)
            if m:
                vi[m.group(1)] = m.group(2)
    return vi

CURRENT = []

def next_chunk():
    """In prompt + lô câu chưa dịch tiếp theo để copy vào web chat."""
    global CURRENT
    vi = load_translations()
    todo = [it for it in scope if it["id"] not in vi]
    print(f"Đã dịch {len(scope) - len(todo)}/{len(scope)} | còn {len(todo)} câu")
    if not todo:
        print("Xong! Chạy tiếp từ cell mục 5.")
        return
    CURRENT = todo[:CHUNK]
    print("=" * 12, f"COPY TỪ DÒNG DƯỚI ({len(CURRENT)} câu)", "=" * 12)
    print(PROMPT + "\n".join(f"{it['id']} | {it['en']}" for it in CURRENT))
    print("=" * 12, "ĐẾN DÒNG TRÊN", "=" * 12)

def save_result(text):
    """Lưu kết quả dán từ web chat; chỉ giữ dòng hợp lệ thuộc lô hiện tại."""
    want = {it["id"] for it in CURRENT}
    text = text.replace("```", "")
    # tách theo id, dùng được cả khi các câu bị dính liền trên một dòng
    pairs = re.findall(
        r"([a-z_]+_\d+)\s*\|\s*(.*?)(?=\s+[a-z_]+_\d+\s*\|\s*|\Z)",
        text, flags=re.S)
    good, bad = {}, []
    for i, t in pairs:
        t = t.strip()
        if i in want and t:
            good[i] = t
        else:
            bad.append(f"{i} | {t}")
    missing = [i for i in want if i not in good]
    if good:
        fn = f"batches_vi/manual_{int(time.time())}.txt"
        with open(fn, "w", encoding="utf-8") as f:
            for i, t in good.items():
                f.write(f"{i} | {t}\n")
    print(f"Lưu {len(good)}/{len(want)} dòng | dòng lạ/hỏng: {len(bad)} | thiếu: {len(missing)}")
    for b in bad[:5]:
        print("  lạ:", b[:100])
    print("Chạy lại next_chunk() để lấy lô tiếp (câu thiếu sẽ tự vào lô sau).")

sẽ dịch 3500 câu


**Vòng lặp:** chạy cell Bước 1, dịch bên web chat, dán kết quả vào cell Bước 2. Lặp lại đến khi xong.

In [96]:
# BƯỚC 1: chạy cell này, copy phần được in ra, dán vào web chat
next_chunk()

Đã dịch 3500/3500 | còn 0 câu
Xong! Chạy tiếp từ cell mục 5.


In [95]:
# BƯỚC 2: dán kết quả từ web chat vào giữa ba dấu nháy, rồi chạy cell này
RESULT = """
test_276 | làm sao để xem ngày đến hạn hóa đơn bug của mình test_358 | làm sao để đổi mã pin tài khoản của mình oos_test_198 | quần denim có dễ phối đồ không test_299 | hóa đơn cáp tháng sau đến hạn khi nào test_562 | khi nào thẻ tín dụng mới của mình sẽ được giao oos_test_59 | trong đợt hạn hán này có được rửa xe không test_340 | đổi mã pin tài khoản tiết kiệm của mình thành 1234 oos_test_33 | những người phản đối vắc xin có tự chết hết nếu không ai tiêm vắc xin cho họ không test_184 | điểm tín dụng của mình là bao nhiêu test_499 | mình muốn biết lãi suất tài khoản thanh toán oos_test_268 | nhẫn đính hôn trung bình có giá bao nhiêu test_10 | tôi muốn chuyển 2 triệu từ tài khoản thanh toán sang tài khoản tiết kiệm test_9 | tôi muốn phân bổ một ít tiền giữa các tài khoản của mình oos_test_272 | xem các tin trên instagram test_351 | vui lòng cho mình biết mã pin tài khoản thanh toán test_520 | tiền thuê nhà và hóa đơn cáp của mình hết bao nhiêu test_425 | hình như có giao dịch mua hàng ở olive garden mà mình không thực hiện trong tài khoản test_589 | mình không mua được đồ ăn bằng thẻ khi ở việt nam oos_train_93 | khi nào bắt đầu học kỳ test_84 | vui lòng tạm khóa tài khoản ngân hàng của mình test_302 | hạn mức thẻ tín dụng visa của mình là bao nhiêu oos_val_36 | thôi miên có hiệu quả trong việc bỏ thói quen xấu không oos_val_15 | tổng thống đến florida lần cuối là khi nào test_597 | tại sao thẻ của mình bị từ chối oos_test_236 | mình đang đi nhanh bao nhiêu oos_test_174 | những tạp chí thể thao nào phổ biến test_53 | số dư trong các tài khoản ngân hàng của mình là bao nhiêu oos_val_89 | trứng có nguồn gốc từ đâu oos_val_23 | năm 2019 có những chương trình truyền hình thực tế mới nào oos_test_143 | có nhà mạng nào cung cấp gói cước di động rẻ hơn không oos_test_274 | mở trang web cnn test_94 | tôi muốn xem tất cả giao dịch của mọi tài khoản trong tháng vừa qua oos_test_243 | liệt kê các cuốn tiểu sử do giáo sư nyu viết sau năm 2001 test_217 | mình có đủ điều kiện mở thẻ tín dụng mới không test_161 | cuối tuần này mình đã tiêu bao nhiêu tiền oos_val_20 | bạn có thể chỉ mình cách tính giá mỗi ounce không test_547 | thẻ ngân hàng đầu tiên của mình hết hạn vào ngày nào test_557 | thẻ citi của mình hết hạn khi nào oos_test_298 | tại thế vận hội olympic, quốc gia nào giành huy chương vàng bóng rổ oos_test_257 | tổ chức các trận đấu sinh tử trong đấu trường tự xây có hợp pháp không test_225 | mình có thể đăng ký thẻ mới ngay tại đây không oos_val_48 | brexit đã chính thức hoàn tất chưa oos_train_75 | những tác phẩm nổi tiếng nhất của van gogh là gì test_392 | dùng thẻ mỹ ở nước ngoài có mất phí giao dịch không test_552 | thẻ tín dụng của mình hết hạn khi nào test_386 | bạn kiểm tra giúp tài khoản của mình có vấn đề gì không, hình như bị khóa rồi test_40 | kiểm tra tài khoản agribank giúp mình xem có đủ tiền mua đôi giày nike này không test_42 | bạn cho mình biết số dư tài khoản ngân hàng được không test_387 | tại sao tài khoản bidv của mình bị tạm giữ oos_test_44 | chỗ sửa mái nhà nào gần đây được đánh giá tốt nhất test_170 | gần đây mình có tiêu quá ngân sách mua sô cô la không oos_train_95 | bao nhiêu phần trăm các loài có đặc tính máu lạnh oos_train_59 | phim shawshank redemption ra mắt năm nào oos_train_51 | bạn giúp mình ôn thi lịch sử được không oos_test_292 | đi chơi vòng đu quay khổng lồ test_233 | cần những điều kiện gì để làm thẻ mastercard test_284 | hóa đơn tiền nước của mình đến hạn khi nào test_540 | mình định mang thẻ đi texas vào tháng 6, lúc đó thẻ còn hạn không oos_test_97 | nước đóng chai ở costco có giá thấp nhất là bao nhiêu oos_test_166 | cho mình xem một số mẫu gọng kính dễ thương oos_val_78 | nên cho mèo ăn bao nhiêu thức ăn oos_val_47 | bạn có thể cho mình biết tin tức mới nhất ở georgia không test_404 | dùng thẻ sacombank ở rome có mất phí giao dịch không test_418 | nếu dùng thẻ ở x thì có bị tính phí không test_221 | làm sao để đăng ký thẻ visa test_586 | mình đang ở úc, định mua đồ ăn vặt mà thẻ bị từ chối thanh toán test_200 | xếp hạng tín dụng của mình là bao nhiêu test_95 | cho mình xem các giao dịch gần đây của tài khoản xyz test_254 | mình cần biết cách báo mất thẻ test_399 | dùng thẻ ở portland có bị tính thêm phí không test_474 | số tiền tối thiểu mình có thể trả cho hóa đơn gas là bao nhiêu test_345 | đặt mã pin tài khoản thanh toán của mình thành 1234 test_203 | cho mình biết điểm tín dụng của mình oos_test_256 | tại sao chúng ta lại bầu người phụ nữ khiến amazon rời đi và mang theo việc làm oos_test_249 | hiện có bao nhiêu thư thoại trong hộp thư đến của mình oos_val_66 | ở pittsburgh có việc làm nào về an ninh mạng không test_308 | hạn mức chi tiêu mỗi ngày của mình là bao nhiêu test_485 | lãi suất tài khoản agribank của mình hiện là bao nhiêu test_133 | mình muốn thanh toán hóa đơn qua ứng dụng test_174 | tháng trước mình đã tiêu bao nhiêu tiền mua bia oos_test_197 | trời mưa thì nên đi giày gì test_588 | tại sao thẻ của mình bị từ chối khi thanh toán bữa tối ở việt nam oos_test_105 | cho mình xem tất cả tiểu sử về teddy kennedy test_72 | tài khoản ngân hàng của mình cần được đóng băng oos_val_27 | robot android có đắt không test_459 | có quy định số tiền thanh toán tối thiểu không oos_test_100 | làm sao để nâng cấp hệ điều hành oos_val_8 | ngân hàng mình có dịch vụ công chứng miễn phí không oos_test_115 | tuyến đường thủy mississippi dài bao nhiêu dặm test_13 | bạn giúp mình chuyển 2 triệu từ tài khoản thanh toán sang tài khoản tiết kiệm được không test_472 | số tiền tối thiểu mình có thể trả cho hóa đơn sưởi là bao nhiêu test_261 | tôi muốn báo thẻ vib bị mất cắp test_532 | hóa đơn điện thoại tháng này của mình hết bao nhiêu test_65 | tôi cần khóa tài khoản ngay bây giờ test_18 | chuyển 1,5 triệu giữa các tài khoản national và citizens oos_train_33 | kobe bryant sinh ngày nào test_371 | làm sao để mở khóa tài khoản ngân hàng oos_train_16 | làm sao để mở hạn mức tín dụng trực tuyến oos_train_37 | utah jazz thi đấu trận tiếp theo khi nào test_219 | cách tốt nhất để đăng ký thẻ mastercard là gì
"""
save_result(RESULT)

Lưu 100/100 dòng | dòng lạ/hỏng: 0 | thiếu: 0
Chạy lại next_chunk() để lấy lô tiếp (câu thiếu sẽ tự vào lô sau).


## 5. Gộp bản dịch và kiểm tra thiếu sót

In [97]:
def load_translations():
    vi = {}
    for fn in sorted(glob.glob("batches_vi/*.txt")):
        for line in open(fn, encoding="utf-8"):
            m = LINE_RE.match(line)
            if m:
                vi[m.group(1)] = m.group(2)
    return vi

vi = load_translations()
missing = [it["id"] for it in scope if it["id"] not in vi]
print("đã dịch:", len([it for it in scope if it["id"] in vi]), "/", len(scope))
print("thiếu:", len(missing), missing[:10])

đã dịch: 3500 / 3500
thiếu: 0 []


Câu bị mô hình bỏ sót sẽ tự được đưa vào lô tiếp theo của `next_chunk()`, nên không cần cell dịch bù riêng. Cell dưới chỉ để kiểm tra còn thiếu bao nhiêu.

In [98]:
missing = [it["id"] for it in scope if it["id"] not in vi]
print("còn thiếu:", len(missing), missing[:10])
if missing:
    print("-> quay lại mục 4, chạy tiếp next_chunk()")

còn thiếu: 0 []


In [100]:
REPLACE = {
    "washington federal": "ocb",
    "barclay's": "tpbank",
    "citibank": "hdbank",
    "discover": "sacombank",
    "amex": "vib",
    "apollo": "shb",
    "western": "eximbank",
}
FIX = {
    "train_381":  "trong kỳ sao kê hiện tại, mình đã chi bao nhiêu tiền ăn uống bằng thẻ visa của bidv",
    "train_1089": "hạn mức tín dụng của thẻ vietcombank của mình là bao nhiêu",
    "train_1345": "phí rút tiền atm bằng thẻ vietcombank ở orlando, florida là bao nhiêu",
    "train_31":   "mình cần chuyển 200k từ tài khoản vietcombank sang tài khoản techcombank",
}

# ---- phần mới thêm: đặt ở đây ----
REPLACE.update({
    "first national": "msb",
    "great western bank": "lpbank",
    "wells fargo": "agribank",
    "wells": "agribank",
    "bb&t": "shinhan bank",
    "td bank": "vietinbank",
    "quicksilver": "techcombank",
    "liên hiệp tín dụng": "quỹ tín dụng nhân dân",
    "walmart": "winmart", "walgreens": "pharmacity", "target": "coopmart",
    "costco": "big c", "safeway": "bách hóa xanh", "albertsons": "bách hóa xanh",
    "trader joe's": "bách hóa xanh", "whole foods": "bách hóa xanh",
    "home depot": "điện máy xanh", "best buy": "thế giới di động",
    "macy's": "vincom", "bloomingdale's": "vincom", "sears": "vincom",
    "zales": "pnj", "fry's": "nguyễn kim", "rosses": "canifa",
    "chipotle": "lotteria", "applebee's": "pizza hut", "boston market": "jollibee",
    "sfo": "sân bay tân sơn nhất","first hawaiian bank": "nam a bank",
    "lloyds": "ncb",
    "sprint": "vinaphone",
    "ucsb": "đại học bách khoa", "payless shoe": "bitis", "amazon":"shoppe","blackjack":"xì dách", "tiger wood" :"Novak Djokovic", "us bank":"ngân hàng nhà nước Việt Nam",
    "jcp":"bách hóa xanh", "scientific american":" khoa học việt nam", "publix":"Aeon","dwight howard":"Quang Hải",
})
FIX.update({
    "train_879": "tôi bị mất thẻ scb và cần báo mất thẻ",
    "train_947": "vé tháng xe buýt của tôi đến hạn thanh toán vào ngày nào",
    "train_948": "bạn có thể cho tôi biết hóa đơn điện evn tháng này khi nào đến hạn không",
    "val_224": "tôi cần cập nhật mã pin thẻ visa",
    "val_297": "có giao dịch trái phép trên thẻ canifa của tôi và tôi cần báo cáo",
    "oos_train_54": "tìm giúp tôi tỉ lệ đá bóng trúng khung thành của Quang Hải"
})
# ---------------------------------

by_id = {it["id"]: it for it in items}
n = 0
for i, t in list(vi.items()):
    ...   # giữ nguyên phần còn lại của cell

## 6. Khử trùng lặp và lưu bộ dữ liệu cuối

Hai câu tiếng Anh khác nhau có thể dịch thành cùng một câu tiếng Việt. Nếu câu đó nằm ở cả train và test thì mô hình đã "thấy đáp án", kết quả sẽ đẹp giả.
Quy tắc: ưu tiên giữ ở tập đứng trước (train > val > test > oos_*), câu trùng ở tập sau bị loại.

In [101]:
import unicodedata

def norm(t):
    """Chuẩn hóa để bắt trùng lặp: NFC, chữ thường, bỏ dấu câu, gộp khoảng trắng."""
    t = unicodedata.normalize("NFC", t).lower()
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

priority = ["train", "val", "test", "oos_train", "oos_val", "oos_test"]
seen, dropped = {}, []          # seen: câu chuẩn hóa -> (tập, nhãn)
final = {k: [] for k in priority}

for split in priority:
    for it in scope:
        if it["split"] != split or it["id"] not in vi:
            continue
        t = unicodedata.normalize("NFC", vi[it["id"]])
        k = norm(t)
        if k in seen:
            dropped.append((t, seen[k], (split, it["label"])))
            continue
        seen[k] = (split, it["label"])
        final[split].append([t, it["label"]])

print("số câu bị loại vì trùng:", len(dropped))
conflict = [x for x in dropped if x[1][1] != x[2][1]]
print("trong đó trùng nhưng KHÁC nhãn (cặp intent dịch ra mơ hồ):", len(conflict))
for x in dropped[:10]:
    print("  ", x)
for x in conflict[:10]:
    print("  KHÁC NHÃN:", x)

json.dump(final, open("subset20_vi.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print({k: len(v) for k, v in final.items()})

số câu bị loại vì trùng: 260
trong đó trùng nhưng KHÁC nhãn (cặp intent dịch ra mơ hồ): 0
   ('mình cần chuyển 200k từ tài khoản vietcombank sang tài khoản techcombank', ('train', 'transfer'), ('train', 'transfer'))
   ('mình muốn đăng ký thẻ tín dụng tích điểm du lịch của vietcombank', ('train', 'new_card'), ('train', 'new_card'))
   ('mình cần đăng ký thẻ tín dụng mới', ('train', 'new_card'), ('train', 'new_card'))
   ('điểm tín dụng của mình là bao nhiêu', ('train', 'credit_score'), ('train', 'credit_score'))
   ('mình muốn đăng ký thẻ tín dụng tích điểm du lịch của vietcombank', ('train', 'new_card'), ('train', 'new_card'))
   ('mình muốn đổi mã pin tài khoản tiết kiệm', ('train', 'pin_change'), ('train', 'pin_change'))
   ('số dư tài khoản tiết kiệm của mình là bao nhiêu', ('train', 'balance'), ('train', 'balance'))
   ('chuyển 100k từ tài khoản tiết kiệm sang tài khoản thanh toán', ('train', 'transfer'), ('train', 'transfer'))
   ('tôi không tìm thấy thẻ mastercard và muốn báo mấ

## 7. Đọc kiểm tra ngẫu nhiên (nên đọc 10-15% số câu, đặc biệt các cặp intent dễ nhầm)

In [102]:
by_id = {it["id"]: it for it in items}
ids = [it["id"] for it in scope if it["id"] in vi]
random.seed(1)
for i in random.sample(ids, min(25, len(ids))):
    it = by_id[i]
    print(f"[{it['label']}]")
    print("  EN:", it["en"])
    print("  VI:", vi[i])

[oos]
  EN: what kind of paint do makeup artists use
  VI: chuyên viên trang điểm dùng loại sơn nào
[expiration_date]
  EN: tell me the expiration date for my current credit card
  VI: cho tôi biết ngày hết hạn của thẻ tín dụng hiện tại
[min_payment]
  EN: what is the minimum i can pay on my heating bill
  VI: số tiền tối thiểu mình có thể trả cho hóa đơn sưởi là bao nhiêu
[transfer]
  EN: send 100 dollars from checking to savings
  VI: chuyển 2 triệu từ tài khoản thanh toán sang tài khoản tiết kiệm
[freeze_account]
  EN: please freeze my account
  VI: vui lòng đóng băng tài khoản của mình
[report_lost_card]
  EN: how do i report that i actually lost my card
  VI: làm sao báo mất thẻ
[pin_change]
  EN: please change my checking account pin to 1234
  VI: vui lòng đổi mã pin tài khoản thanh toán thành 1234
[expiration_date]
  EN: is my card set to expire soon
  VI: thẻ của tôi sắp hết hạn phải không
[expiration_date]
  EN: when is my credit card set to expire
  VI: thẻ tín dụng của tôi d

## 8. Việc tiếp theo

- Khi chất lượng ổn: đặt `TEST_MODE = False` ở cell cấu hình, chạy lại từ cell mục 4 (các lô đã xong được giữ nguyên, `next_chunk()` chỉ lấy câu còn thiếu).
- Tải `subset20_vi.json` và thư mục `batches_vi/` về từ panel Tệp.
- Ghi lại cho báo cáo: web chat/model dùng để dịch, prompt, tỉ lệ câu đã đọc kiểm tra, số câu sửa tay, số câu bị loại vì trùng (và số câu trùng khác nhãn).
- Tạo thêm tập test riêng: câu rút gọn (thiếu thông tin), câu đa ý (ghép 2 câu khác intent), near-OOS.

In [104]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [105]:
import os, shutil

DST = "/content/drive/MyDrive/clinc20_vi"      # thư mục đích, tự tạo nếu chưa có
os.makedirs(DST, exist_ok=True)

# các file đơn
for fn in ["data_full.json", "items_en.json", "subset20_en.json",
           "subset20_en_full.json", "subset20_vi.json"]:
    if os.path.exists(fn):
        shutil.copy(fn, DST)

# thư mục bản dịch (copy đè lên bản cũ nếu có)
if os.path.exists("batches_vi"):
    shutil.copytree("batches_vi", f"{DST}/batches_vi", dirs_exist_ok=True)

print(os.listdir(DST))

['data_full.json', 'items_en.json', 'subset20_en.json', 'subset20_en_full.json', 'subset20_vi.json', 'batches_vi']
